# Pipeline de modelo predictivo y visualización
**Caso A: minería de asteroides** · Challenge 1 · Machine Learning · UPCH

> **ADVERTENCIA:** todos los datos de este notebook son **SINTÉTICOS (simulados)** y se generan con supuestos propios. No son observaciones reales ni provienen de la infografía. Las predicciones **no representan la realidad** ni demuestran viabilidad o inviabilidad económica.

## 1. Definición de la tarea de aprendizaje
**Tarea:** regresión supervisada. Predecir un **indicador económico simulado** (valor neto simulado de una misión, en unidades arbitrarias, u.a.) a partir de variables de misión simuladas.

**Por qué datos sintéticos:** la infografía no reporta costos, ingresos ni rentabilidad de misiones; solo dos valores de mercado (2026, 2030), un CAGR y un conteo de lanzamientos. Con eso no se puede entrenar un modelo económicamente significativo. Se hace un **experimento didáctico**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Datos de la infografía (los únicos factuales) ---
info = pd.DataFrame({"año": [2026, 2030], "mercado_USD_mil_millones": [2.49, 5.42]})
cagr_infografia = 0.214
cagr_implicito = (5.42 / 2.49) ** (1 / 4) - 1
print(info)
print("(La pequeña diferencia se explica por el redondeo de los valores del gráfico.)")
print(f"CAGR informado: {cagr_infografia:.1%} | CAGR implícito en los dos valores: {cagr_implicito:.1%}")
print("Solo hay 2 puntos de mercado: no alcanzan para entrenar un modelo, y una proyección de")
print("mercado NO equivale a la rentabilidad de una misión minera.")

## 2. Preparación de los datos (sintéticos)
**Variables simuladas** (rangos supuestos, sin base en la infografía):

| Variable | Distribución supuesta |
|---|---|
| `costo_mision_ua` | Uniforme(100, 500) |
| `concentracion` | Uniforme(0.1, 1.0) |
| `prob_exito` | Uniforme(0.1, 0.9) |
| `tiempo_retorno_anios` | Uniforme(3, 20) |
| `incert_regulatoria` | Uniforme(0, 1) |

**Objetivo simulado:** `van_ua = prob_exito · concentración · K / (1 + tasa)^tiempo − costo · (1 + 0,3 · incert_regulatoria) + ruido`.
`K` (escala de ingresos) y `tasa` son supuestos. El caso base usa `K = 3000` para no inclinar el resultado a priori; luego se analiza la sensibilidad a `K`.

In [ ]:
def generar_datos(n=800, K=3000.0, tasa=0.08, sigma=40.0, semilla=42):
    rng = np.random.default_rng(semilla)
    df = pd.DataFrame({
        "costo_mision_ua": rng.uniform(100, 500, n),
        "concentracion": rng.uniform(0.1, 1.0, n),
        "prob_exito": rng.uniform(0.1, 0.9, n),
        "tiempo_retorno_anios": rng.uniform(3, 20, n),
        "incert_regulatoria": rng.uniform(0, 1, n),
    })
    ingreso = df.prob_exito * df.concentracion * K / (1 + tasa) ** df.tiempo_retorno_anios
    costo = df.costo_mision_ua * (1 + 0.3 * df.incert_regulatoria)
    df["van_ua"] = ingreso - costo + rng.normal(0, sigma, n)
    return df

df = generar_datos()
print("DATOS SINTÉTICOS (simulados), n =", len(df))
display(df.describe().round(2))
print("Valores nulos:", int(df.isna().sum().sum()))

FEATURES = ["costo_mision_ua", "concentracion", "prob_exito", "tiempo_retorno_anios", "incert_regulatoria"]
X, y = df[FEATURES], df["van_ua"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)
print("Entrenamiento:", X_tr.shape, "| Prueba:", X_te.shape)

## 3. Selección y justificación del modelo de regresión
Se comparan dos modelos:

- **Regresión lineal** (con estandarización): línea base interpretable; sus coeficientes indican el efecto de cada variable. Puede quedarse corta si hay interacciones o no linealidades.
- **Random Forest Regressor**: captura interacciones y no linealidades sin especificarlas, a costa de menor interpretabilidad (se compensa con importancia por permutación).

**Criterio:** mayor R² medio en validación cruzada de 5 particiones sobre el conjunto de entrenamiento; el conjunto de prueba se usa una sola vez al final.

Nota: como los datos los genera una fórmula que conocemos, el modelo solo puede *recuperar nuestros supuestos*; no descubre nada sobre el mundo real.

In [ ]:
modelos = {
    "Regresión lineal": Pipeline([("sc", StandardScaler()), ("m", LinearRegression())]),
    "Random Forest": Pipeline([("sc", StandardScaler()),
                               ("m", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))]),
}
cv = {n: cross_val_score(m, X_tr, y_tr, cv=5, scoring="r2") for n, m in modelos.items()}
cv_df = pd.DataFrame({n: {"R2 medio (CV)": s.mean(), "desv. est.": s.std()} for n, s in cv.items()}).T
display(cv_df.round(3))
mejor = cv_df["R2 medio (CV)"].idxmax()
print("Modelo seleccionado por validación cruzada:", mejor)

## 4. Entrenamiento y evaluación
Se entrena el modelo elegido con todo el entrenamiento y se evalúa en prueba.

In [ ]:
modelo = modelos[mejor].fit(X_tr, y_tr)
pred = modelo.predict(X_te)
metricas = {"MAE (u.a.)": mean_absolute_error(y_te, pred),
            "RMSE (u.a.)": mean_squared_error(y_te, pred) ** 0.5,
            "R2": r2_score(y_te, pred)}
print(f"Modelo: {mejor}  (datos sintéticos)")
for k, v in metricas.items():
    print(f"{k}: {v:.3f}")

# Lineal como referencia, para ver cuánto aporta la no linealidad
lin = modelos["Regresión lineal"].fit(X_tr, y_tr)
print(f"R2 en prueba, regresión lineal (referencia): {r2_score(y_te, lin.predict(X_te)):.3f}")
modelo = modelos[mejor].fit(X_tr, y_tr)

## 5. Visualización e interpretación
Tres paneles: (a) valores simulados vs predichos, (b) importancia de variables por permutación, (c) **sensibilidad del supuesto de ingresos `K`**: proporción de misiones simuladas con valor neto positivo al variar `K`.

In [ ]:
imp = permutation_importance(modelo, X_te, y_te, n_repeats=20, random_state=42, scoring="r2")
orden = np.argsort(imp.importances_mean)

Ks = np.linspace(500, 8000, 16)
frac = [(generar_datos(n=2000, K=k, semilla=7).van_ua > 0).mean() for k in Ks]

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].scatter(y_te, pred, alpha=0.5, s=14)
lim = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
ax[0].plot(lim, lim, "r--", label="predicción perfecta")
ax[0].set_xlabel("Valor neto simulado (u.a.)"); ax[0].set_ylabel("Predicho (u.a.)")
ax[0].set_title(f"(a) {mejor}: real vs predicho\n(DATOS SINTÉTICOS)"); ax[0].legend()

ax[1].barh(np.array(FEATURES)[orden], imp.importances_mean[orden], xerr=imp.importances_std[orden])
ax[1].set_xlabel("Caída de R2 al permutar"); ax[1].set_title("(b) Importancia de variables\n(refleja la fórmula supuesta)")

ax[2].plot(Ks, frac, "o-")
ax[2].axvline(3000, color="gray", ls=":", label="K del caso base")
ax[2].set_xlabel("Supuesto de escala de ingresos K (u.a.)")
ax[2].set_ylabel("Proporción con valor neto > 0")
ax[2].set_title("(c) El resultado depende del supuesto K"); ax[2].legend()
plt.tight_layout()
os.makedirs("imagenes", exist_ok=True)
plt.savefig("imagenes/prediccion_resultados.png", dpi=140, bbox_inches="tight")
plt.show()
print("Proporción de misiones simuladas con valor neto > 0, según K:")
print(pd.DataFrame({"K": Ks.round(0), "proporción > 0": np.round(frac, 3)}).to_string(index=False))

## 6. Conclusiones y limitaciones
**Interpretación**
- El panel (a) y las métricas indican qué tan bien el modelo recupera la fórmula que *nosotros* definimos; la importancia (b) refleja esa misma fórmula.
- El panel (c) es el resultado clave: la proporción de misiones con valor positivo pasa de muy pocas a la mayoría solo variando el supuesto de ingresos `K` (ver tabla impresa). Es decir, **sin datos reales de ingresos y costos, el signo de la rentabilidad lo decide el supuesto, no el modelo**.

**Conclusión alineada con la postura del proyecto:** la infografía muestra avances tecnológicos y crecimiento proyectado del mercado, pero no aporta costos, ingresos, riesgos ni plazos de retorno. Por ello, la viabilidad económica actual **no está demostrada**. Este experimento no la refuta: solo muestra que, para evaluarla, harían falta los datos que la infografía no entrega.

**Limitaciones**
- Datos 100 % sintéticos; unidades arbitrarias; distribuciones y fórmula supuestas.
- Hay circularidad: el modelo aprende una fórmula escrita por el autor.
- Los resultados no son predicciones sobre la minería de asteroides real.
- El crecimiento de mercado (CAGR) no es una medida de rentabilidad de una misión.
- Datos necesarios para un análisis real: costos de misión y extracción, cantidad y concentración del recurso, precios y demanda, probabilidad de éxito, plazo de retorno, costo del capital y marco regulatorio.